# EDA Summary & Dataset Preparation — 50 Indian Cities

This notebook summarizes EDA results, finalizes feature selection, identifies cities that require exclusion, defines a leakage-safe train/validation/test strategy, and exports datasets and reports for Feature Engineering.

Dataset: https://www.kaggle.com/datasets/mukeshdevrath007/indian-5000-cities-weather-data

## Before running

In Kaggle, click **Add Input** and attach `mukeshdevrath007/indian-5000-cities-weather-data`. Run every cell in order.

The notebook processes one hourly city file at a time and retains only daily summaries, avoiding memory errors.

In [1]:
import os
import re
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

SAVE = "/kaggle/working/eda_summary_feature_engineering/"
os.makedirs(SAVE, exist_ok=True)

print("Setup complete")

Setup complete


## Step 1 — Find Kaggle CSV files

In [2]:
INPUT_ROOT = Path("/kaggle/input")
if not INPUT_ROOT.exists():
    raise FileNotFoundError("Run this notebook inside Kaggle.")

csv_files = sorted([
    str(file) for file in INPUT_ROOT.rglob("*")
    if file.is_file() and (
        file.name.lower().endswith(".csv") or
        file.name.lower().endswith(".csv.gz")
    )
])

if not csv_files:
    raise FileNotFoundError(
        "No CSV files found. Attach the weather dataset using Add Input."
    )

print(f"CSV files found: {len(csv_files):,}")
print("First file:", csv_files[0])

CSV files found: 4,686
First file: /kaggle/input/datasets/mukeshdevrath007/indian-5000-cities-weather-data/Weather_Data_Scraping_and_Analysis/weather.csv


## Step 2 — Prepare the same deterministic city candidates

In [3]:
def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def get_city_name(file_path):
    path = Path(file_path)
    name = path.name[:-7] if path.name.lower().endswith(".csv.gz") else path.stem
    generic = {"weather", "weatherdata", "hourlyweather", "data"}
    return path.parent.name if normalize_name(name) in generic else name


available_files = pd.DataFrame({
    "city": [get_city_name(file) for file in csv_files],
    "file": csv_files,
})
available_files["city_key"] = available_files["city"].apply(normalize_name)

invalid_words = ["weatherdata", "scraping", "analysis", "readme", "metadata"]
available_files = available_files[
    ~available_files["city_key"].apply(
        lambda name: any(word in name for word in invalid_words)
    )
].drop_duplicates("city_key").sort_values("city").reset_index(drop=True)

PREFERRED_CITIES = [
    "Agartala", "Agra", "Ahmedabad", "Aizawl", "Amritsar",
    "Bhopal", "Chandigarh", "Chennai", "Coimbatore", "Cuttack",
    "Dehra Dun", "Delhi", "Guwahati", "Hyderabad", "Imphal",
    "Indore", "Itanagar", "Jaipur", "Jammu", "Jodhpur",
    "Kochi", "Kohima", "Kolkata", "Kota", "Leh",
    "Lucknow", "Ludhiana", "Mumbai", "Nagpur", "Patna",
    "Prayagraj", "Pune", "Raipur", "Rajkot", "Ranchi"
]
preferred_keys = {normalize_name(city) for city in PREFERRED_CITIES}
preferred = available_files[available_files["city_key"].isin(preferred_keys)]
remaining = available_files[
    ~available_files["city_key"].isin(preferred["city_key"])
].reset_index(drop=True)

extra_count = min(70, len(remaining))
indices = np.linspace(0, len(remaining) - 1, extra_count, dtype=int)
candidate_files = pd.concat([preferred, remaining.iloc[indices]], ignore_index=True)

print("Preferred cities found:", len(preferred))
print("Candidate files prepared:", len(candidate_files))

Preferred cities found: 35
Candidate files prepared: 105


## Step 3 — Convert each hourly city file into daily features

In [4]:
COLUMN_ALIASES = {
    "time": ["time", "datetime", "date"],
    "temperature_2m": ["temperature2m", "temp2m", "temperature"],
    "relative_humidity_2m": ["relativehumidity2m", "humidity2m", "relativehumidity"],
    "dew_point_2m": ["dewpoint2m", "dewpoint"],
    "precipitation": ["precipitation", "precip"],
    "rain": ["rain"],
    "cloud_cover": ["cloudcover", "cloudiness"],
    "cloud_cover_low": ["cloudcoverlow", "lowcloudcover"],
    "cloud_cover_mid": ["cloudcovermid", "midcloudcover"],
    "cloud_cover_high": ["cloudcoverhigh", "highcloudcover"],
    "wind_speed_10m": ["windspeed10m", "wind10m"],
    "wind_speed_100m": ["windspeed100m", "wind100m"],
    "wind_gusts_10m": ["windgusts10m", "windgust10m", "windgusts"],
    "pressure_msl": ["pressuremsl", "sealevelpressure"],
    "surface_pressure": ["surfacepressure"],
}


def match_columns(columns):
    normalized = {column: normalize_name(column) for column in columns}
    matched = {}
    for standard, aliases in COLUMN_ALIASES.items():
        targets = [normalize_name(alias) for alias in aliases]
        exact = [column for column, name in normalized.items() if name in targets]
        prefix = [column for column, name in normalized.items()
                  if any(name.startswith(target) for target in targets)]
        if exact or prefix:
            matched[standard] = (exact or prefix)[0]
    return matched


def summarize_city(file, city):
    header = pd.read_csv(file, nrows=0).columns.tolist()
    matched = match_columns(header)
    if "time" not in matched or "temperature_2m" not in matched:
        raise ValueError("Required time or temperature column not found")

    raw = pd.read_csv(file, usecols=list(dict.fromkeys(matched.values())), low_memory=False)
    raw = raw.rename(columns={original: standard for standard, original in matched.items()})
    raw["datetime"] = pd.to_datetime(raw.pop("time"), errors="coerce", utc=True).dt.tz_localize(None)
    raw = raw.dropna(subset=["datetime"])
    if raw.empty:
        raise ValueError("No valid timestamps")

    for column in raw.columns.difference(["datetime"]):
        raw[column] = pd.to_numeric(raw[column], errors="coerce", downcast="float")

    raw["date"] = raw["datetime"].dt.floor("D")
    grouped = raw.groupby("date", sort=False)
    daily = pd.DataFrame(index=grouped.size().index)

    def add(source, outputs):
        if source in raw:
            for output, operation in outputs.items():
                daily[output] = grouped[source].agg(operation)

    add("temperature_2m", {"temp_mean": "mean", "temp_max": "max", "temp_min": "min"})
    add("relative_humidity_2m", {"humidity_mean": "mean"})
    add("dew_point_2m", {"dew_mean": "mean"})
    add("precipitation", {"precip_total": "sum", "precip_max": "max"})
    if "precip_total" not in daily:
        add("rain", {"precip_total": "sum", "precip_max": "max"})
    add("cloud_cover", {"cloud_mean": "mean"})
    add("cloud_cover_low", {"cloud_low_mean": "mean"})
    add("cloud_cover_mid", {"cloud_mid_mean": "mean"})
    add("cloud_cover_high", {"cloud_high_mean": "mean"})
    add("wind_speed_10m", {"wind_mean_10m": "mean", "wind_max_10m": "max"})
    add("wind_speed_100m", {"wind_mean_100m": "mean", "wind_max_100m": "max"})
    add("wind_gusts_10m", {"wind_gusts_max": "max"})
    add("pressure_msl", {"pressure_mean": "mean"})
    add("surface_pressure", {"surface_pressure_mean": "mean"})

    if "precip_total" in daily:
        daily["rain_occurred"] = (daily["precip_total"] > 0).astype("int8")

    daily = daily.reset_index()
    daily.insert(1, "city", city)

    raw_duplicate_timestamps = int(raw["datetime"].duplicated().sum())
    del raw
    gc.collect()
    return daily, matched, raw_duplicate_timestamps

In [5]:
daily_frames = []
load_records = []
loaded_cities = set()

for _, row in candidate_files.iterrows():
    if len(loaded_cities) >= 50:
        break
    if row["city"] in loaded_cities:
        continue

    try:
        city_daily, matched, raw_duplicates = summarize_city(row["file"], row["city"])
        daily_frames.append(city_daily)
        loaded_cities.add(row["city"])
        load_records.append({
            "city": row["city"], "status": "loaded",
            "daily_rows": len(city_daily), "matched_columns": len(matched),
            "raw_duplicate_timestamps": raw_duplicates, "error": ""
        })
    except Exception as error:
        load_records.append({
            "city": row["city"], "status": "skipped", "daily_rows": 0,
            "matched_columns": 0, "raw_duplicate_timestamps": 0,
            "error": str(error)[:160]
        })

    if len(load_records) % 10 == 0:
        print(f"Checked {len(load_records)} files; loaded {len(loaded_cities)}/50 cities")

load_log = pd.DataFrame(load_records)
display(load_log)

if not daily_frames:
    raise RuntimeError("No valid city files loaded. Review load_log.")

df = pd.concat(daily_frames, ignore_index=True)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

print("Final shape:", df.shape)
print("Cities loaded:", df["city"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

if df["city"].nunique() < 50:
    print("Warning: fewer than 50 valid cities loaded; reports use the available cities.")

Checked 10 files; loaded 10/50 cities
Checked 20 files; loaded 20/50 cities
Checked 30 files; loaded 30/50 cities
Checked 40 files; loaded 40/50 cities
Checked 50 files; loaded 50/50 cities


,city,status,daily_rows,matched_columns,raw_duplicate_timestamps,error
0,Agartala,loaded,5164,15,0,
1,Agra,loaded,5164,15,0,
2,Ahmedabad,loaded,5164,15,0,
3,Aizawl,loaded,5164,15,0,
4,Amritsar,loaded,5164,15,0,
5,Bhopal,loaded,5164,15,0,
6,Chandigarh,loaded,5164,15,0,
7,Chennai,loaded,5164,15,0,
8,Coimbatore,loaded,5164,15,0,
9,Cuttack,loaded,5164,15,0,


Final shape: (258200, 23)
Cities loaded: 50
Date range: 2010-01-01 to 2024-02-20


## Step 4 — Summarize overall EDA findings

In [6]:
weather_features = [
    column for column in df.select_dtypes(include=np.number).columns
    if column not in ["year", "month"]
]

duplicate_city_days = int(df.duplicated(["city", "date"]).sum())
missing_cells = int(df[weather_features].isna().sum().sum())
missing_pct = missing_cells / (len(df) * len(weather_features)) * 100

eda_summary = pd.DataFrame({
    "metric": [
        "cities_loaded", "daily_rows", "weather_features", "start_date",
        "end_date", "duplicate_city_days", "missing_weather_cells",
        "overall_missing_pct"
    ],
    "value": [
        df["city"].nunique(), len(df), len(weather_features),
        str(df["date"].min().date()), str(df["date"].max().date()),
        duplicate_city_days, missing_cells, round(missing_pct, 3)
    ]
})
display(eda_summary)

,metric,value
0,cities_loaded,50
1,daily_rows,258200
2,weather_features,19
3,start_date,2010-01-01
4,end_date,2024-02-20
5,duplicate_city_days,0
6,missing_weather_cells,0
7,overall_missing_pct,0.000


## Step 5 — Identify insufficient cities

In [7]:
city_quality_rows = []
raw_duplicate_map = (load_log[load_log["status"] == "loaded"]
                     .set_index("city")["raw_duplicate_timestamps"].to_dict())

for city, part in df.groupby("city"):
    first_date = part["date"].min()
    last_date = part["date"].max()
    expected_days = (last_date - first_date).days + 1
    observed_days = part["date"].nunique()
    city_weather = part[weather_features]

    violations = 0
    checked = 0
    if {"temp_max", "temp_mean"}.issubset(part.columns):
        violations += int((part["temp_max"] < part["temp_mean"]).sum())
        checked += len(part)
    if {"temp_mean", "temp_min"}.issubset(part.columns):
        violations += int((part["temp_mean"] < part["temp_min"]).sum())
        checked += len(part)
    if "humidity_mean" in part:
        violations += int(((part["humidity_mean"] < 0) | (part["humidity_mean"] > 100)).sum())
        checked += len(part)
    if "precip_total" in part:
        violations += int((part["precip_total"] < 0).sum())
        checked += len(part)

    city_quality_rows.append({
        "city": city, "first_date": first_date, "last_date": last_date,
        "years_covered": part["year"].nunique(), "observed_days": observed_days,
        "expected_days": expected_days, "coverage_pct": observed_days / expected_days * 100,
        "missing_pct": city_weather.isna().mean().mean() * 100,
        "duplicate_city_days": int(part.duplicated(["city", "date"]).sum()),
        "raw_duplicate_timestamps": raw_duplicate_map.get(city, 0),
        "consistency_violation_pct": violations / checked * 100 if checked else 0,
    })

city_quality = pd.DataFrame(city_quality_rows)
city_quality["exclude_city"] = (
    (city_quality["coverage_pct"] < 90) |
    (city_quality["years_covered"] < 10) |
    (city_quality["missing_pct"] > 20) |
    (city_quality["duplicate_city_days"] > 0) |
    (city_quality["consistency_violation_pct"] > 1)
)

def exclusion_reason(row):
    reasons = []
    if row["coverage_pct"] < 90: reasons.append("coverage below 90%")
    if row["years_covered"] < 10: reasons.append("fewer than 10 years")
    if row["missing_pct"] > 20: reasons.append("missingness above 20%")
    if row["duplicate_city_days"] > 0: reasons.append("duplicate city-days")
    if row["consistency_violation_pct"] > 1: reasons.append("consistency violations above 1%")
    return "; ".join(reasons) if reasons else "include"

city_quality["reason"] = city_quality.apply(exclusion_reason, axis=1)
excluded_cities = city_quality[city_quality["exclude_city"]].copy()
included_cities = city_quality[~city_quality["exclude_city"]].copy()

display(city_quality.sort_values(["exclude_city", "coverage_pct"], ascending=[False, True]))
print("Cities included:", len(included_cities))
print("Cities excluded:", len(excluded_cities))

,city,first_date,last_date,years_covered,observed_days,expected_days,coverage_pct,missing_pct,duplicate_city_days,raw_duplicate_timestamps,consistency_violation_pct,exclude_city,reason
0,Abbigeri,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
1,Agartala,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
2,Agra,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
3,Ahmedabad,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
4,Aizawl,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
5,Alampalaiyam,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
6,Amingaon,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
7,Amritsar,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
8,Antarvedipalem,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include
9,Atmakur_4,2010-01-01,2024-02-20,15,5164,5164,100.000,0.000,0,0,0.000,False,include


Cities included: 50
Cities excluded: 0


## Step 6 — Finalize feature selection

In [8]:
feature_quality = pd.DataFrame({
    "feature": weather_features,
    "missing_pct": [df[column].isna().mean() * 100 for column in weather_features],
    "unique_values": [df[column].nunique(dropna=True) for column in weather_features],
    "variance": [df[column].var() for column in weather_features],
})

corr = df[weather_features].corr().abs()
np.fill_diagonal(corr.values, np.nan)
feature_quality["max_abs_correlation"] = feature_quality["feature"].map(corr.max())
feature_quality["most_correlated_with"] = feature_quality["feature"].map(corr.idxmax())

# Core variables are protected because they represent distinct business meanings.
protected_features = {
    "temp_mean", "temp_max", "temp_min", "precip_total", "rain_occurred"
}

# Prefer one representative from known redundant groups.
preferred_representatives = {
    "wind_mean_10m": "wind_mean_100m",
    "wind_max_10m": "wind_max_100m",
    "surface_pressure_mean": "pressure_mean",
    "cloud_low_mean": "cloud_mean",
    "cloud_mid_mean": "cloud_mean",
    "cloud_high_mean": "cloud_mean",
}

def feature_decision(row):
    feature = row["feature"]
    replacement = preferred_representatives.get(feature)
    if row["missing_pct"] > 30:
        return "EXCLUDE", "missingness above 30%"
    if row["unique_values"] <= 1 or row["variance"] == 0:
        return "EXCLUDE", "constant or zero-variance feature"
    if replacement in weather_features:
        return "EXCLUDE", f"use {replacement} as representative"
    if row["max_abs_correlation"] >= 0.98 and feature not in protected_features:
        return "REVIEW", "near-duplicate correlation"
    return "SELECT", "useful weather signal"

decisions = feature_quality.apply(feature_decision, axis=1)
feature_quality["decision"] = [item[0] for item in decisions]
feature_quality["reason"] = [item[1] for item in decisions]

selected_features = feature_quality.loc[
    feature_quality["decision"].isin(["SELECT", "REVIEW"]), "feature"
].tolist()
excluded_features = feature_quality.loc[
    feature_quality["decision"] == "EXCLUDE", "feature"
].tolist()

display(feature_quality.sort_values(["decision", "max_abs_correlation"]))
print("Selected/review features:", selected_features)
print("Excluded features:", excluded_features)

,feature,missing_pct,unique_values,variance,max_abs_correlation,most_correlated_with,decision,reason
17,surface_pressure_mean,0.000,227637,2794.318,0.514,temp_mean,EXCLUDE,use pressure_mean as representative
10,cloud_high_mean,0.000,2401,1340.138,0.798,cloud_mean,EXCLUDE,use cloud_mean as representative
8,cloud_low_mean,0.000,2390,356.112,0.817,cloud_mean,EXCLUDE,use cloud_mean as representative
9,cloud_mid_mean,0.000,2401,458.483,0.830,cloud_mean,EXCLUDE,use cloud_mean as representative
12,wind_max_10m,0.000,3232,31.484,0.914,wind_max_100m,EXCLUDE,use wind_max_100m as representative
11,wind_mean_10m,0.000,256090,17.889,0.971,wind_mean_100m,EXCLUDE,use wind_mean_100m as representative
3,humidity_mean,0.000,253547,338.086,0.636,cloud_mean,SELECT,useful weather signal
18,rain_occurred,0.000,2,0.249,0.699,cloud_mean,SELECT,useful weather signal
16,pressure_mean,0.000,8767,36.473,0.789,temp_min,SELECT,useful weather signal
4,dew_mean,0.000,159866,57.455,0.812,temp_min,SELECT,useful weather signal


Selected/review features: ['temp_mean', 'temp_max', 'temp_min', 'humidity_mean', 'dew_mean', 'precip_total', 'precip_max', 'cloud_mean', 'wind_mean_100m', 'wind_max_100m', 'wind_gusts_max', 'pressure_mean', 'rain_occurred']
Excluded features: ['cloud_low_mean', 'cloud_mid_mean', 'cloud_high_mean', 'wind_mean_10m', 'wind_max_10m', 'surface_pressure_mean']


## Step 7 — Prepare the clean Feature Engineering dataset

In [9]:
included_city_names = included_cities["city"].tolist()

prepared_df = df[df["city"].isin(included_city_names)].copy()
prepared_df = (prepared_df.drop_duplicates(["city", "date"])
               .sort_values(["city", "date"]).reset_index(drop=True))

final_columns = ["city", "date", "year", "month"] + selected_features
final_columns = [column for column in final_columns if column in prepared_df]
prepared_df = prepared_df[final_columns]

print("Prepared dataset shape:", prepared_df.shape)
print("Prepared cities:", prepared_df["city"].nunique())
print("Final columns:", prepared_df.columns.tolist())
display(prepared_df.head())

Prepared dataset shape: (258200, 17)
Prepared cities: 50
Final columns: ['city', 'date', 'year', 'month', 'temp_mean', 'temp_max', 'temp_min', 'humidity_mean', 'dew_mean', 'precip_total', 'precip_max', 'cloud_mean', 'wind_mean_100m', 'wind_max_100m', 'wind_gusts_max', 'pressure_mean', 'rain_occurred']


,city,date,year,month,temp_mean,temp_max,temp_min,humidity_mean,dew_mean,precip_total,precip_max,cloud_mean,wind_mean_100m,wind_max_100m,wind_gusts_max,pressure_mean,rain_occurred
0,Abbigeri,2010-01-01,2010,1,24.228,28.720,19.370,63.530,16.453,0.000,0.000,21.929,18.105,25.772,35.640,1012.729,0
1,Abbigeri,2010-01-02,2010,1,23.170,28.420,18.069,61.533,14.976,0.000,0.000,1.012,17.425,23.032,31.680,1014.883,0
2,Abbigeri,2010-01-03,2010,1,22.417,28.319,16.970,59.659,13.701,0.000,0.000,3.325,10.682,16.677,25.920,1014.592,0
3,Abbigeri,2010-01-04,2010,1,23.040,29.170,16.819,51.590,11.880,0.000,0.000,1.288,6.771,12.229,24.480,1012.533,0
4,Abbigeri,2010-01-05,2010,1,23.613,29.420,16.069,47.436,11.144,0.000,0.000,6.825,8.704,21.708,20.520,1010.954,0


## Step 8 — Define chronological train, validation and test splits

Weather data is temporal, so random splitting would leak future patterns into training. The split is chronological:

- Train: data before the final three calendar years
- Validation: third-last calendar year
- Test: final two calendar years

Every city follows the same date boundaries.

In [10]:
available_years = sorted(prepared_df["year"].dropna().astype(int).unique())
if len(available_years) < 4:
    raise ValueError("At least four years are required for chronological splitting.")

validation_year = available_years[-3]
test_years = available_years[-2:]

prepared_df["data_split"] = np.select(
    [
        prepared_df["year"] < validation_year,
        prepared_df["year"] == validation_year,
        prepared_df["year"].isin(test_years),
    ],
    ["train", "validation", "test"],
    default="exclude"
)

train_df = prepared_df[prepared_df["data_split"] == "train"].copy()
validation_df = prepared_df[prepared_df["data_split"] == "validation"].copy()
test_df = prepared_df[prepared_df["data_split"] == "test"].copy()

split_summary = prepared_df.groupby("data_split").agg(
    rows=("date", "size"),
    cities=("city", "nunique"),
    start_date=("date", "min"),
    end_date=("date", "max")
).reset_index()
split_summary["row_pct"] = split_summary["rows"] / len(prepared_df) * 100

print("Validation year:", validation_year)
print("Test years:", test_years)
display(split_summary)

assert train_df["date"].max() < validation_df["date"].min(), "Train/validation leakage found"
assert validation_df["date"].max() < test_df["date"].min(), "Validation/test leakage found"
print("Chronological leakage check: PASSED")

Validation year: 2022
Test years: [np.int64(2023), np.int64(2024)]


,data_split,rows,cities,start_date,end_date,row_pct
0,test,20800,50,2023-01-01,2024-02-20,8.056
1,train,219150,50,2010-01-01,2021-12-31,84.876
2,validation,18250,50,2022-01-01,2022-12-31,7.068


Chronological leakage check: PASSED


## Step 9 — Final preparation checks

In [11]:
final_checks = pd.DataFrame({
    "check": [
        "duplicate city-date rows", "missing city values", "missing date values",
        "train rows", "validation rows", "test rows", "included cities",
        "excluded cities", "selected features"
    ],
    "result": [
        prepared_df.duplicated(["city", "date"]).sum(),
        prepared_df["city"].isna().sum(), prepared_df["date"].isna().sum(),
        len(train_df), len(validation_df), len(test_df),
        prepared_df["city"].nunique(), len(excluded_cities), len(selected_features)
    ],
    "status": [
        "PASS" if prepared_df.duplicated(["city", "date"]).sum() == 0 else "REVIEW",
        "PASS" if prepared_df["city"].isna().sum() == 0 else "REVIEW",
        "PASS" if prepared_df["date"].isna().sum() == 0 else "REVIEW",
        "PASS" if len(train_df) > 0 else "FAIL",
        "PASS" if len(validation_df) > 0 else "FAIL",
        "PASS" if len(test_df) > 0 else "FAIL",
        "PASS", "PASS", "PASS" if selected_features else "FAIL"
    ]
})
display(final_checks)

,check,result,status
0,duplicate city-date rows,0,PASS
1,missing city values,0,PASS
2,missing date values,0,PASS
3,train rows,219150,PASS
4,validation rows,18250,PASS
5,test rows,20800,PASS
6,included cities,50,PASS
7,excluded cities,0,PASS
8,selected features,13,PASS


## Step 10 — Export datasets and reports

In [12]:
eda_summary.to_csv(SAVE + "eda_summary.csv", index=False)
load_log.to_csv(SAVE + "load_log.csv", index=False)
city_quality.to_csv(SAVE + "city_quality_report.csv", index=False)
included_cities.to_csv(SAVE + "included_cities.csv", index=False)
excluded_cities.to_csv(SAVE + "excluded_cities.csv", index=False)
feature_quality.to_csv(SAVE + "feature_selection_report.csv", index=False)
split_summary.to_csv(SAVE + "train_validation_test_summary.csv", index=False)
final_checks.to_csv(SAVE + "final_preparation_checks.csv", index=False)

prepared_df.to_csv(SAVE + "feature_engineering_ready_all.csv", index=False)
train_df.to_csv(SAVE + "feature_engineering_train.csv", index=False)
validation_df.to_csv(SAVE + "feature_engineering_validation.csv", index=False)
test_df.to_csv(SAVE + "feature_engineering_test.csv", index=False)

print("All reports and datasets saved in:", SAVE)
print("Download them from the Kaggle Output panel after execution.")

All reports and datasets saved in: /kaggle/working/eda_summary_feature_engineering/
Download them from the Kaggle Output panel after execution.


## Final interpretation

- City exclusions are based on explicit coverage, duration, missingness, duplication and consistency thresholds.
- Feature exclusions remove high-missing, constant and known redundant variables while protecting important temperature and rainfall features.
- The split is chronological to prevent future weather information from leaking into model training.
- The exported datasets are daily, sorted by city and date, and ready for feature engineering such as lags, rolling statistics, season labels and cyclical time features.